# Why does tasmin's disaggregation stall when every other variable finishes?

**Thesis (falsifiable):** the disaggregation code is identical for all variables, but the
*input to it* is **eager** for regular variables and **lazy** for tasmin — and that one
difference decides whether the shard write is cheap slicing or an all-to-all rechunk that
blows up memory.

- **Regular variables** (`tas`, `tasmax`, `pr`, `rsds`, `dtr`): `fit_historical` builds the
  debiased-coarse field via `_apply_bias_correction`, which runs ibicus on `.as_numpy()`
  inputs and returns `xr.DataArray(data=debiased_np)` — a **numpy-backed (eager)** array
  (`pipeline.py:736-756`). So `_apply_spatial_downscaling` runs the `slinear` interp
  **eagerly**, and the trailing `.chunk(SHARD)` merely wraps concrete data → the write is a
  cheap per-shard slice.
- **tasmin**: `fit_historical_tasmin` builds its input as
  `derive_tasmin(_open_from_icechunk(tasmax), _open_from_icechunk(dtr))` — a **lazy dask**
  graph opened with `chunks="auto"` (which collapses time to a single block). The interp +
  `.chunk(SHARD)` therefore stay a lazy graph, deferred to the write, where the interp's
  full-lat block must be rechunked into the 72-row shard grid: an all-to-all rechunk that
  re-derives / holds the whole ~56-129 GB fine array → the production stall.

This notebook demonstrates the difference with the **real** `downscale_from_coarse` at
production shapes, using synthetic random data (interp/rechunk cost depends on *shape*, not
values). Run on a Coiled VM. `slinear` is deliberate (issue #77 — it removes negative
artifacts that plain `linear` introduces), so the point here is the *graph shape*, not the
interpolation method.

In [ ]:
import multiprocessing
import shutil
import tempfile
import time

import dask
import dask.array as darr
import numpy as np
import xarray as xr

from srm.downscaling_utils import downscale_from_coarse, interpolate_coarse_to_fine_grid
from srm.encoding import SHARD_LAT, SHARD_LON, SHARD_TIME, make_encoding

print("cores:", multiprocessing.cpu_count())
print("shard grid:", SHARD_TIME, SHARD_LAT, SHARD_LON)


# ── grids ────────────────────────────────────────────────────────────────────
COARSE_LAT, COARSE_LON = 192, 288  # CESM2-WACCM coarse grid
FINE_LAT, FINE_LON = 721, 1440  # ERA5 0.25 deg
NT_OBS = 3 * 365  # obs series feeding the doy climatology (short on purpose)
rng = np.random.default_rng(0)

obs_time = xr.date_range("1980-01-01", periods=NT_OBS, freq="D", use_cftime=True)
clat = np.linspace(-89, 89, COARSE_LAT)
clon = np.linspace(0, 358.75, COARSE_LON)
flat = np.linspace(-89.875, 89.875, FINE_LAT)
flon = np.linspace(0, 359.75, FINE_LON)

# obs are ALWAYS eager numpy in production (_apply_spatial_downscaling calls .as_numpy())
obs_coarse = xr.DataArray(
    rng.random((NT_OBS, COARSE_LAT, COARSE_LON), dtype="float32"),
    dims=("time", "lat", "lon"),
    coords={"time": obs_time, "lat": clat, "lon": clon},
    name="v",
)
obs_fine = xr.DataArray(
    rng.random((NT_OBS, FINE_LAT, FINE_LON), dtype="float32"),
    dims=("time", "lat", "lon"),
    coords={"time": obs_time, "lat": flat, "lon": flon},
    name="v",
)
SHARD = {"time": SHARD_TIME, "lat": SHARD_LAT, "lon": SHARD_LON}


def coarse_debiased(nt, *, lazy):
    """The disaggregation input. lazy=False mimics the eager numpy array that
    _apply_bias_correction returns (regular vars); lazy=True mimics tasmin's
    derive_tasmin(_open_from_icechunk(...)) — dask, chunks='auto' single time block."""
    t = xr.date_range("1978-01-01", periods=nt, freq="D", use_cftime=True)
    if lazy:
        data = darr.random.random(
            (nt, COARSE_LAT, COARSE_LON), chunks=(nt, COARSE_LAT, COARSE_LON)
        ).astype("float32")
    else:
        data = rng.random((nt, COARSE_LAT, COARSE_LON), dtype="float32")  # numpy -> eager
    return xr.DataArray(
        data, dims=("time", "lat", "lon"), coords={"time": t, "lat": clat, "lon": clon}, name="v"
    )


def disagg(da):
    """Exactly what _apply_spatial_downscaling does: downscale then chunk to the shard grid."""
    d = downscale_from_coarse(da, obs_coarse, obs_fine, method="additive", clim_method="fft")
    return d.chunk(SHARD).rename("v")


def graph_size(da):
    return len(dict(da.data.__dask_graph__())) if da.chunks is not None else 0

## 1. The one difference — eager (regular) vs lazy (tasmin) input

Small grid here so the eager array builds instantly; the point is the *nature* of the input.

In [ ]:
eager_in = coarse_debiased(1825, lazy=False)
lazy_in = coarse_debiased(1825, lazy=True)
print(
    "regular var input (from _apply_bias_correction): chunks =",
    eager_in.chunks,
    "-> EAGER numpy" if eager_in.chunks is None else "",
)
print(
    "tasmin input (derive_tasmin from store)        : chunks =",
    {d: (len(c), c[0]) for d, c in zip(lazy_in.dims, lazy_in.chunks)},
    "-> LAZY dask, one time block",
)

## 2. What that does to the shard-write graph

`disagg()` is the real `downscale_from_coarse(...).chunk(SHARD)`. Compare the size of the
dask graph the write must execute for each input. (Small grid: `fine 216x432` = a few lat
shards, so the topology is present but cheap to build.)

In [ ]:
# temporarily shrink the fine grid so the eager disagg builds fast
_FL, _FLo = FINE_LAT, FINE_LON
FINE_LAT, FINE_LON = 216, 432
flat_s = np.linspace(-89, 89, FINE_LAT)
flon_s = np.linspace(0, 359, FINE_LON)
obs_fine_s = xr.DataArray(
    rng.random((NT_OBS, FINE_LAT, FINE_LON), dtype="float32"),
    dims=("time", "lat", "lon"),
    coords={"time": obs_time, "lat": flat_s, "lon": flon_s},
    name="v",
)


def disagg_small(da):
    d = downscale_from_coarse(da, obs_coarse, obs_fine_s, method="additive", clim_method="fft")
    return d.chunk(SHARD).rename("v")


eager_out = disagg_small(coarse_debiased(1825, lazy=False))
lazy_out = disagg_small(coarse_debiased(1825, lazy=True))
print(
    f"EAGER (regular) write graph : {graph_size(eager_out):>6} tasks  <- concrete data, per-shard slices"
)
print(
    f"LAZY  (tasmin)  write graph : {graph_size(lazy_out):>6} tasks  <- interp + groupby + all-to-all rechunk"
)
print(
    f"ratio: {graph_size(lazy_out) / max(graph_size(eager_out), 1):.0f}x more tasks for the lazy (tasmin) path"
)
FINE_LAT, FINE_LON = _FL, _FLo  # restore

## 3. The smoking gun — one output shard's dependencies

Cull the graph for a **single** 72x144 output shard. Eager: one concrete slice. Lazy: it
pulls the **full-lat interp** (every task), because the interp collapsed lat into one block —
this is the all-to-all rechunk that forces the whole fine array resident under memory pressure.

In [ ]:
_FL, _FLo = FINE_LAT, FINE_LON
FINE_LAT, FINE_LON = 216, 432
for label, lazy in [("EAGER (regular)", False), ("LAZY  (tasmin) ", True)]:
    out = disagg_small(coarse_debiased(1825, lazy=lazy))
    one_shard = out.isel(lat=slice(0, SHARD_LAT), lon=slice(0, SHARD_LON))
    full = graph_size(out)
    shard = len(dict(one_shard.data.__dask_graph__()))
    print(f"{label}: one 72x144 shard pulls {shard:>5} tasks   (whole array = {full} tasks)")
FINE_LAT, FINE_LON = _FL, _FLo

## 4. Timing — eager vs lazy at production shapes, swept by scale

Materialize each path to a local temp zarr (the real shard encoding) at the full `721x1440`
grid. The eager path is what regular variables run; the lazy path is tasmin's. Watch the ratio
**grow with scale** — that divergence is the stall in the making. `eager GB` is the peak the
eager path holds; both fit the production VMs. Bump `SCALES` toward 13514 (the real span) on
a big VM to approach the production magnitude.

In [ ]:
def materialize(da):
    tmp = tempfile.mkdtemp(prefix="disagg_")
    try:
        disagg(da).to_dataset().to_zarr(
            tmp, mode="w", encoding=make_encoding("v"), zarr_format=3, consolidated=False
        )
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


SCALES = [1825, 3650, 7300]
hdr = f"{'days':>6} {'GB':>6} {'eager(s)':>9} {'lazy(s)':>9} {'lazy/eager':>11}"
print(hdr)
print("-" * len(hdr))
with dask.config.set(scheduler="threads"):
    for nt in SCALES:
        gb = FINE_LAT * FINE_LON * nt * 4 / 1e9
        t0 = time.perf_counter()
        materialize(coarse_debiased(nt, lazy=False))
        te = time.perf_counter() - t0
        t0 = time.perf_counter()
        materialize(coarse_debiased(nt, lazy=True))
        tl = time.perf_counter() - t0
        print(f"{nt:6d} {gb:6.1f} {te:9.1f} {tl:9.1f} {tl / te:10.1f}x")

## 5. The `slinear` interp does not scale with cores

Independent of eager/lazy: the coarse→fine `slinear` interp is GIL-bound, so even the eager
path's up-front compute (and any lazy recompute) runs at ~1-2 cores no matter the VM size.

In [ ]:
resid = (
    coarse_debiased(7300, lazy=True).groupby("time.dayofyear")
    - obs_coarse.groupby("time.dayofyear").mean()
)


def interp_only():
    return float(interpolate_coarse_to_fine_grid(resid, obs_fine).sum().compute())


res = {}
for n in [1, 2, 4, 8, 16, 32]:
    if n > multiprocessing.cpu_count():
        break
    with dask.config.set(scheduler="threads", num_workers=n):
        t0 = time.perf_counter()
        interp_only()
        res[n] = time.perf_counter() - t0
        print(f"interp threads={n:2d}: {res[n]:6.1f}s   speedup {res[1] / res[n]:4.1f}x")

## 6. (Optional) real production chunking — tasmin's actual lazy input

Grounds the synthetic `lazy=True` in reality: open the live production store the way
`fit_historical_tasmin` does and show the debiased-coarse tasmax/dtr chunking. Needs S3 read
(the Coiled instance profile has it). Skips cleanly if the store/branch is unavailable.

In [ ]:
try:
    import icechunk

    from srm.config import _icechunk_storage_for_path

    STORE = "s3://carbonplan-srm/output/production/CESM2-WACCM-ERA5-global.icechunk"
    repo = icechunk.Repository.open(_icechunk_storage_for_path(STORE))
    session = repo.readonly_session(branch="v0.10.0")
    for var in ("tasmax", "dtr"):
        ds = xr.open_dataset(
            session.store,
            engine="zarr",
            consolidated=False,
            chunks="auto",
            group=f"debiased_coarse/historical/{var}/001",
        )
        da = ds[var]
        print(
            f"prod debiased_coarse {var}: dims={dict(da.sizes)} "
            f"chunks={{{', '.join(f'{d}:{len(c)}x{c[0]}' for d, c in zip(da.dims, da.chunks))}}}"
        )
    print("\n-> time collapses to ONE block (chunks='auto'); this is the real lazy tasmin input.")
except Exception as exc:
    print("skipped (store unavailable):", type(exc).__name__, exc)

## Summary for the issue

| | regular variables | tasmin |
|---|---|---|
| debiased-coarse input | **eager numpy** (`_apply_bias_correction` → `xr.DataArray(data=np)`) | **lazy dask** (`derive_tasmin(_open_from_icechunk(...))`, `chunks="auto"`) |
| interp executes | eagerly, up front | deferred into the write |
| `.chunk(SHARD)` | wraps concrete data (cheap slices) | all-to-all rechunk of the full fine array |
| write graph | ~n_shards tasks | interp + groupby + rechunk (orders of magnitude more) |
| one output shard | 1 concrete slice | pulls the full-lat interp |
| peak memory | one full-array hold (fits) | full array **plus** rechunk working set → 400 GB hang |

Plus the tasmin-only tax that compounds it: it is serialized **after** tasmax+dtr, adds a
**reconcile** pass (extra full-array read + swap), and **writes its store twice** (raw →
reconciled). `slinear` itself is not the bug (issue #77 chose it deliberately) and does not
parallelize (Section 5); the fix is to stop deferring the interp behind a lazy all-to-all
rechunk — e.g. materialize tasmin's disaggregation eagerly **before** `.chunk(SHARD)`, the way
every regular variable already does.